# Доступный анализ текста (spaCy · TF-IDF · KMeans)

Полный конвейер текстовой аналитики с поддержкой **польского, русского, английского, итальянского, финского и исландского** языков.  
Все результаты выводятся через `print()` — без цветовой кодировки и псевдографики.  
Совместим с экранными читалками **NVDA** и **JAWS**.

**Единственное, что нужно от пользователя:** задать путь к файлу `.pdf`, `.txt` или `.docx` в первой ячейке кода. Если файл не найден — используется встроенный пример.

---

**Этапы анализа:**

1. **Загрузка корпуса** — PDF постранично (pdfplumber), TXT, DOCX (python-docx), HTML-файл или **URL** (прямой адрес страницы); BeautifulSoup удаляет навигацию, боковые панели и блоки «Похожие статьи» при загрузке из URL или HTML; встроенный пример как запасной вариант.
2. **Определение языка** — `langdetect` голосует по всем документам и выбирает доминирующий язык.
3. **Языковая модель spaCy** — автоматически выбирается модель версии **Large (lg)**: `pl_core_news_lg`, `ru_core_news_lg`, `en_core_web_lg`, `it_core_news_lg`, `fi_core_news_lg`; для исландского (`is`) — `spacy.blank("is")` с базовой токенизацией; при отсутствии модели — базовый токенизатор с sentencizer. Модели **lg** обеспечивают значительно более высокое качество тематизации (KMeans) и распознавания именованных сущностей (NER).
4. **Коррекция текста** — удаление точных повторов слов (все языки); нормализация аббревиатур с пробелами для польского (`m. in.`→`m.in.`), русского (`т. е.`→`т.е.`) и английского (`e. g.`→`e.g.`).
5. **Базовый NLP** — токенизация на слова и предложения, фильтрация стоп-слов, лемматизация, POS-теггинг (NOUN/VERB/ADJ/…).
6. **Именованные сущности (NER)** — люди, организации, места, даты и другие объекты (с частотной сводкой по всему корпусу).
7. **Мешок слов (CountVectorizer)** — матрица частот с языко-специфичными стоп-словами; топ-10 слов и топ-5 биграмм.
8. **TF-IDF и автопоиск** — запрос автоматически строится из топ-терминов корпуса; косинусное сходство гарантированно > 0.
9. **Структура текста** — весь корпус объединяется в поток, делится на предложения (spaCy) и группируется в абзацы по 3–6 предложений.
10. **Ключевые слова** — рейтинговая таблица терминов и фраз (1–3 слова) по средневзвешенному TF-IDF на уровне абзацев.
11. **Тезисы** — лучшее по TF-IDF-сумме предложение каждого абзаца; сохраняются в `тезисы.txt` в формате `- предложение`.
12. **Тематизация (KMeans)** — кластеризация абзацев по spaCy-векторам; для каждой темы — ключевые слова и пример фрагмента *(требует модели `*_md` или `*_lg` с векторами)*.
13. **Сентимент-анализ** — многоязычная модель `nlptown/bert-base-multilingual-uncased-sentiment` (шкала 1–5 звёзд); при недоступности — автопереключение на английскую модель.
14. **Экспорт результатов** — CSV (UTF-8 BOM, совместимо с Excel), JSON (ключевые слова тем), TXT (тезисы) в папку `export_results/`.
15. **Итоговый отчёт** — все ключевые метрики в одном текстовом блоке.

In [ ]:
# ============================================================
# ВВОД ДАННЫХ: путь к исходному файлу/URL берётся из config.json.
# Скопируйте config.example.json в config.json и укажите source_file.
# Если config.json отсутствует — используется встроенный пример.
# ============================================================
import os, re, json
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse

SOURCE_FILE = ""
CUSTOM_PATTERNS = []

_CONFIG_PATH = Path("config.json")
if _CONFIG_PATH.is_file():
    try:
        _cfg = json.loads(_CONFIG_PATH.read_text(encoding="utf-8"))
        SOURCE_FILE = (_cfg.get("source_file") or "").strip()
        CUSTOM_PATTERNS = list(_cfg.get("custom_patterns") or [])
        print(f"[OK] Конфигурация загружена: {_CONFIG_PATH}")
    except Exception as _e:
        print(f"[ВНИМАНИЕ] Не удалось прочитать config.json: {_e}")
else:
    print("[ИНФО] Файл config.json не найден — используется встроенный пример.")
    print("       Скопируйте config.example.json в config.json и задайте source_file.")

def clean_web_article_start(text, check_window=400):
    """Remove common web-scraping artifacts from extracted article text:
    - Photo credits/view counters: "Fot. CPK 521", standalone "CPK 524"
    - Doubled metadata from responsive layouts: "Author Date Author Date"
    """
    if not text:
        return text
    # Remove photo credit patterns (Polish "Fot." = photo)
    text = re.sub(r"\bFot\.\s+[A-Z][A-Za-z0-9]*\.?\s*\d*\s*", " ", text)
    # Remove ALL-CAPS word + 3-digit number that leaked in as view/photo counter
    text = re.sub(r"\b[A-Z]{2,}\s+\d{3,}\s+", " ", text)
    # Remove doubled short phrases in the first check_window chars
    # (responsive design often repeats title/author/date for mobile+desktop)
    head = text[:check_window]
    m = re.search(r"(.{10,60})\s+\1", head)
    if m:
        repeated = re.escape(m.group(1))
        text = re.sub(r"(" + repeated + r")\s+\1", r"\1", text, count=1)
    return re.sub(r"\s+", " ", text).strip()

def clean_pdf_text(s: str) -> str:
    """Базовая очистка текста из PDF: мягкие дефисы, разорванные слова, пробелы."""
    if not s:
        return ""
    s = s.replace("\u00ad", "")              # мягкий дефис
    s = re.sub(r"(\w)-\n(\w)", r"\1\2", s)  # склейка слов через перенос
    s = s.replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _norm_line(line: str) -> str:
    """Нормализация строки для сравнения: убираем ведущие/замыкающие цифры."""
    s = re.sub(r"^\s*\d+\s*", "", line.strip())
    s = re.sub(r"\s*\d+\s*$", "", s.strip())
    return re.sub(r"\s+", " ", s).lower().strip()

def strip_repeated_headers(raw_pages, min_fraction=0.25, max_line_chars=150):
    """Удаляет повторяющиеся колонтитулы.
    Строки, встречающиеся (после удаления цифр) в >= min_fraction страниц,
    считаются колонтитулами и удаляются из всех страниц.
    """
    threshold = max(2, int(len(raw_pages) * min_fraction))
    line_counts = Counter()
    for page in raw_pages:
        seen = set()
        for line in page.splitlines():
            if line.strip() and len(line.strip()) <= max_line_chars:
                n = _norm_line(line)
                if len(n) > 5 and n not in seen:
                    line_counts[n] += 1
                    seen.add(n)
    boilerplate = {n for n, c in line_counts.items() if c >= threshold}
    cleaned = []
    for page in raw_pages:
        kept = [line for line in page.splitlines()
                if _norm_line(line) not in boilerplate]
        cleaned.append("\n".join(kept))
    return cleaned, boilerplate

def load_pdf(path):
    """Загружает PDF постранично, удаляет повторяющиеся колонтитулы, очищает текст."""
    try:
        import pdfplumber
    except ImportError:
        print("[ВНИМАНИЕ] pdfplumber не установлен. Выполните: pip install pdfplumber")
        return []
    raw_pages, skipped = [], 0
    with pdfplumber.open(path) as pdf:
        total = len(pdf.pages)
        for page in pdf.pages:
            text = page.extract_text() or ""
            if text.strip():
                raw_pages.append(text)
            else:
                skipped += 1
    # Удаляем повторяющиеся колонтитулы до объединения строк
    cleaned_raw, removed_bp = strip_repeated_headers(raw_pages)
    if removed_bp:
        print(f"  Удалено {len(removed_bp)} повторяющихся колонтитулов:")
        for bp in sorted(removed_bp)[:5]:
            print(f"    - '{bp[:70]}'")
        if len(removed_bp) > 5:
            print(f"    ... и ещё {len(removed_bp)-5}")
    # Применяем финальную очистку
    pages = [p for p in (clean_pdf_text(r) for r in cleaned_raw) if p]
    print(f"  PDF: {total} страниц, загружено {len(pages)}, пропущено {skipped}.")
    return pages

def load_txt(path, page_size=2000):
    """Загружает текстовый файл и разбивает на псевдостраницы."""
    with open(path, encoding="utf-8") as f:
        raw = f.read()
    pages, text = [], raw.strip()
    # Применяем CUSTOM_PATTERNS
    if CUSTOM_PATTERNS:
        for _cp in CUSTOM_PATTERNS:
            text = re.sub(_cp, "", text)
        text = re.sub(r"\\s+", " ", text).strip()
    while len(text) > page_size:
        split_at = text.rfind(" ", 0, page_size)
        if split_at == -1:
            split_at = page_size
        pages.append(text[:split_at].strip())
        text = text[split_at:].strip()
    if text:
        pages.append(text)
    print(f"  TXT: разбито на {len(pages)} псевдостраниц (~{page_size} символов каждая).")
    return pages


def load_docx(path, page_size=2000):
    """Загружает файл .docx и извлекает текст из всех абзацев документа."""
    try:
        from docx import Document
    except ImportError:
        print("[ВНИМАНИЕ] python-docx не установлен. Выполните: pip install python-docx")
        return []
    doc = Document(path)
    raw = "  ".join(p.text for p in doc.paragraphs if p.text.strip())
    raw = re.sub(r"\s+", " ", raw).strip()
    # Применяем CUSTOM_PATTERNS (если определены)
    if 'CUSTOM_PATTERNS' in globals():
        for pat in CUSTOM_PATTERNS:
            raw = re.sub(pat, "", raw)
    pages, text = [], raw
    while len(text) > page_size:
        split_at = text.rfind(" ", 0, page_size)
        if split_at == -1:
            split_at = page_size
        pages.append(text[:split_at].strip())
        text = text[split_at:].strip()
    if text:
        pages.append(text)
    print(f"  DOCX: извлечено {len(raw)} символов, "
          f"разбито на {len(pages)} псевдостраниц (~{page_size} символов каждая).")
    return pages

# ============================================================
# Стоп-фразы для отсечения блоков «Похожие статьи» / «Read also» и т.п.
# Покрывают все поддерживаемые языки: pl, ru, en, it, fi, is.
# ============================================================
RELATED_ARTICLES_STOPWORDS = [
    # Polski
    "podobne artykuły", "podobne wpisy", "powiązane artykuły", "powiązane wpisy",
    "polecane artykuły", "polecamy", "czytaj też", "czytaj także",
    "również cię zainteresuje", "może cię zainteresować", "zobacz również",
    "więcej z", "więcej na ten temat",
    # Русский
    "похожие статьи", "похожие материалы", "подобные статьи",
    "читайте также", "смотрите также", "вам также может понравиться",
    "рекомендуем", "ещё по теме",
    # English
    "related articles", "related posts", "see also", "read also",
    "you may also like", "you might also like", "may also like",
    "more from", "more like this", "trending now",
    "recommended for you", "you might enjoy",
    # Italiano
    "articoli correlati", "leggi anche", "potrebbe interessarti anche",
    "scopri di più", "ti potrebbe interessare", "altri articoli",
    # Suomi
    "aiheeseen liittyviä artikkeleita", "lue myös", "lisää aiheesta",
    "saatat pitää myös", "katso myös", "lisää aiheeseen liittyen",
    # Íslenska
    "tengdar greinar", "lesa meira", "sjá einnig", "tengt efni",
    "þér gæti einnig líkað",
]

def load_html(path, page_size=2000):
    """Извлекает основной текст из HTML-файла через BeautifulSoup.
    Удаляет: навигацию, шапку/подвал, боковые панели, блоки
    «Похожие статьи» / «Podobne artykuły» и тексты ссылок-меню.
    """
    try:
        from bs4 import BeautifulSoup
    except ImportError:
        print("[ВНИМАНИЕ] beautifulsoup4 не установлен: pip install beautifulsoup4 lxml")
        return []
    with open(path, encoding="utf-8", errors="replace") as _f:
        html = _f.read()
    soup = BeautifulSoup(html, "html.parser")
    # 1. Remove tags that never contain article text
    for tag in soup(["script","style","nav","header","footer","aside",
                      "noscript","form","button","iframe","figure"]):
        tag.decompose()
    # 2. Find the main content container
    main = (soup.find("main") or soup.find("article")
            or soup.find(id=lambda x: x and any(k in x.lower()
                         for k in ("content","article","main","post","story")))
            or soup.find(class_=lambda x: x and any(k in " ".join(x).lower()
                         for k in ("entry-content","post-content","article-body",
                                   "article-content","story-body"))))
    if not main:
        # last resort: largest block
        candidates = [el for el in soup.find_all(["article","section","div"])
                       if el.get_text(strip=True)]
        main = max(candidates, key=lambda e: len(e.get_text()), default=soup)
    # 3. Remove nav/boilerplate elements INSIDE main (not globally)
    _NAV_INNER = ["related","podobne","powiązane","comment","komentarz",
                  "social","share","advertisement","reklama","banner","popup",
                  "widget","subscribe","newsletter","sidebar","correlati",
                  "liittyv","tengd"]
    for el in list(main.find_all(True)):
        if not el.attrs:
            continue
        _cls = " ".join(el.get("class") or []).lower()
        _id  = (el.get("id") or "").lower()
        if any(p in _cls or p in _id for p in _NAV_INNER):
            el.decompose()
    # 4. Cut off "Podobne artykuły" / "Related articles" sections
    for hdr in main.find_all(["h1","h2","h3","h4","h5","p","div","section"]):
        if any(sw in hdr.get_text().lower() for sw in RELATED_ARTICLES_STOPWORDS):
            for sib in list(hdr.next_siblings):
                sib.decompose() if hasattr(sib, "decompose") else None
            hdr.decompose()
            break
    # 5. Extract, normalise and clean web artifacts
    raw_text = re.sub(r"\s+", " ", main.get_text(separator=" ", strip=True)).strip()
    raw_text = clean_web_article_start(raw_text)
    if not raw_text:
        print("[ВНИМАНИЕ] Не удалось извлечь текст из HTML.")
        return []
    pages = []
    text = raw_text
    while len(text) > page_size:
        split_at = text.rfind(" ", 0, page_size)
        if split_at == -1:
            split_at = page_size
        pages.append(text[:split_at].strip())
        text = text[split_at:].strip()
    if text:
        pages.append(text)
    print(f"  HTML: извлечено {len(raw_text)} символов, "
          f"разбито на {len(pages)} псевдостраниц (~{page_size} символов каждая).")
    return pages

def load_url(url, page_size=2000):
    """Загружает страницу по URL, очищает её BeautifulSoup и разбивает на псевдостраницы.
    Применяет ту же логику очистки, что и load_html():
    удаляет навигацию, боковые панели, блоки «Похожие статьи» и тексты ссылок-меню.
    """
    try:
        import requests
        from bs4 import BeautifulSoup
    except ImportError:
        print("[ВНИМАНИЕ] requests или beautifulsoup4 не установлены.")
        print("  pip install requests beautifulsoup4 lxml")
        return []
    print(f"  Загрузка URL: {url}")
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=20)
        resp.raise_for_status()
        resp.encoding = resp.apparent_encoding
        html = resp.text
    except Exception as _e:
        print(f"[ВНИМАНИЕ] Не удалось загрузить URL: {_e}")
        return []
    soup = BeautifulSoup(html, "html.parser")
    # 1. Remove tags that never contain article text
    for tag in soup(["script","style","nav","header","footer","aside",
                      "noscript","form","button","iframe","figure"]):
        tag.decompose()
    # 2. Find the main content container
    main = (soup.find("main") or soup.find("article")
            or soup.find(id=lambda x: x and any(k in x.lower()
                         for k in ("content","article","main","post","story")))
            or soup.find(class_=lambda x: x and any(k in " ".join(x).lower()
                         for k in ("entry-content","post-content","article-body",
                                   "article-content","story-body"))))
    if not main:
        candidates = [el for el in soup.find_all(["article","section","div"])
                       if el.get_text(strip=True)]
        main = max(candidates, key=lambda e: len(e.get_text()), default=soup)
    # 3. Remove nav/boilerplate elements INSIDE main (not globally)
    _NAV_INNER = ["related","podobne","powiązane","comment","komentarz",
                  "social","share","advertisement","reklama","banner","popup",
                  "widget","subscribe","newsletter","sidebar","correlati",
                  "liittyv","tengd"]
    for el in list(main.find_all(True)):
        if not el.attrs:
            continue
        _cls = " ".join(el.get("class") or []).lower()
        _id  = (el.get("id") or "").lower()
        if any(p in _cls or p in _id for p in _NAV_INNER):
            el.decompose()
    # 4. Cut off "Podobne artykuły" / "Related articles" sections
    for hdr in main.find_all(["h1","h2","h3","h4","h5","p","div","section"]):
        if any(sw in hdr.get_text().lower() for sw in RELATED_ARTICLES_STOPWORDS):
            for sib in list(hdr.next_siblings):
                sib.decompose() if hasattr(sib, "decompose") else None
            hdr.decompose()
            break
    # 5. Extract, normalise and clean web artifacts
    raw_text = re.sub(r"\s+", " ", main.get_text(separator=" ", strip=True)).strip()
    raw_text = clean_web_article_start(raw_text)
    if not raw_text:
        print("[ВНИМАНИЕ] Не удалось извлечь текст со страницы.")
        return []
    pages, text = [], raw_text
    while len(text) > page_size:
        split_at = text.rfind(" ", 0, page_size)
        if split_at == -1:
            split_at = page_size
        pages.append(text[:split_at].strip())
        text = text[split_at:].strip()
    if text:
        pages.append(text)
    print(f"  URL: извлечено {len(raw_text)} символов, "
          f"разбито на {len(pages)} псевдостраниц (~{page_size} символов каждая).")
    return pages

FALLBACK_CORPUS = [
    """I have a significant problem with Word. I am typing a document and suddenly
    it will turn to gibberish, not readable or recognisable, just random symbols.
    One time I was able to get it back but I have just lost an important document.
    When I preview the document in search I can read what I wrote, but as soon as
    I open it it's gone. Please help, I am a writer and can't risk losing my work!""",
    """Machine learning is a field of artificial intelligence that uses statistical
    techniques to give computer systems the ability to learn from data.""",
    """The document processing system encountered multiple errors during the batch
    operation. Files were corrupted and recovery attempts failed.""",
]

corpus = []
if SOURCE_FILE and SOURCE_FILE.startswith(("http://", "https://")):
    corpus = load_url(SOURCE_FILE)
elif SOURCE_FILE and os.path.isfile(SOURCE_FILE):
    ext = os.path.splitext(SOURCE_FILE)[1].lower()
    print(f"Загрузка из файла: {SOURCE_FILE}")
    if ext == ".pdf":
        corpus = load_pdf(SOURCE_FILE)
    elif ext in (".txt", ".text"):
        corpus = load_txt(SOURCE_FILE)
    elif ext == ".docx":
        corpus = load_docx(SOURCE_FILE)
    elif ext in (".html", ".htm"):
        corpus = load_html(SOURCE_FILE)
    else:
        print(f"[ВНИМАНИЕ] Неизвестное расширение: {ext}. Поддерживаются: .pdf, .txt, .html, .docx")
elif SOURCE_FILE:
    print(f"[ВНИМАНИЕ] Источник не найден: {SOURCE_FILE!r}. Используется встроенный пример.")

_used_fallback = False
if not corpus:
    print("Используется встроенный корпус-пример (3 документа).")
    corpus = FALLBACK_CORPUS
    _used_fallback = True

corpus = [c for c in corpus if c.strip()]
_N = len(corpus)
print(f"\nКорпус готов: {_N} документ(ов).")
_show = min(5, _N)
for i in range(_show):
    _prev = corpus[i].strip().replace("\n", " ")[:80]
    print(f"  [{i+1}] {_prev}...")
if _N > _show:
    print(f"  ... и ещё {_N - _show} документ(ов) (показаны первые {_show})")

# ============================================================
# PROJECT_DIR: подкаталог в export_results, индивидуальный для источника.
# Для файлов: имя файла без расширения. Для URL: домена_слаг.
# Для встроенного корпуса: _default. Внутри подкаталога файлы перезаписываются.
# ============================================================
def _slugify(s, maxlen=80):
    s = re.sub(r"[^\w\-\.]+", "_", s, flags=re.UNICODE).strip("._")
    return s[:maxlen] or "_default"

if not SOURCE_FILE or not corpus or _used_fallback:
    PROJECT_NAME = "_default"
elif SOURCE_FILE.startswith(("http://", "https://")):
    _u = urlparse(SOURCE_FILE)
    _host = (_u.netloc or "url").replace("www.", "")
    _path = _u.path.strip("/").replace("/", "_") or "index"
    PROJECT_NAME = _slugify(f"{_host}_{_path}")
else:
    PROJECT_NAME = _slugify(Path(SOURCE_FILE).stem)

PROJECT_DIR = Path("export_results") / PROJECT_NAME
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nКаталог проекта: {PROJECT_DIR}")


## 1) Определение языка

Библиотека `langdetect` анализирует статистику символов и n-грамм, чтобы определить язык текста.  
Мы запускаем детектор на каждом документе, затем выбираем самый часто встречающийся язык — это повышает надёжность.  
Поддерживаемые языки: **польский** (`pl`), **русский** (`ru`), **английский** (`en`), **итальянский** (`it`), **финский** (`fi`), **исландский** (`is`).

> Если корпус содержит смешанные языки, будет выбран доминирующий.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from langdetect import detect, DetectorFactory, LangDetectException
from collections import Counter

DetectorFactory.seed = 42

SUPPORTED_LANGS = {"en", "pl", "ru", "it", "fi", "is"}
LANG_NAMES = {"en": "английский", "pl": "польский", "ru": "русский", "it": "итальянский", "fi": "финский", "is": "исландский"}

def detect_corpus_language(texts):
    """Returns (winner_lang, outlier_indices).
    outlier_indices — set of 0-based document indices detected as a
    language OTHER than the corpus-dominant winner.
    """
    detections = []
    for t in texts:
        try:
            detections.append(detect(t[:500]))
        except LangDetectException:
            detections.append(None)
    valid = [l for l in detections if l]
    if not valid:
        print("[ВНИМАНИЕ] Язык не определён. Используется английский.")
        return "en", set()
    counts = Counter(valid)
    winner = "en"
    for lang, _ in counts.most_common():
        if lang in SUPPORTED_LANGS:
            winner = lang
            break
    outliers = {i for i, l in enumerate(detections) if l and l != winner}
    if outliers:
        print(f"Преобладающий язык: {LANG_NAMES.get(winner, winner)} ({winner})")
        print(f"Нетипичных страниц (другой язык, игнорируются): {len(outliers)}")
        for idx in sorted(outliers)[:10]:
            print(f"  Документ {idx+1}: определён как '{detections[idx]}' "
                  f"(вероятно, иноязычный фрагмент или упражнение)")
        if len(outliers) > 10:
            print(f"  ... и ещё {len(outliers)-10}")
    else:
        print(f"Язык определён единогласно: "
              f"{LANG_NAMES.get(winner, winner)} ({winner})")
    print(f"Всего документов: {len(texts)}, типичных: "
          f"{len(valid)-len(outliers)}, нетипичных: {len(outliers)}")
    print(f"Все определения: {dict(counts)}")
    return winner, outliers

print("--- Определение языка ---")
LANG, LANG_OUTLIER_PAGES = detect_corpus_language(corpus)
print(f"\n[РЕЗУЛЬТАТ] Выбран язык: {LANG_NAMES.get(LANG, LANG)} ({LANG})")
if LANG_OUTLIER_PAGES:
    print(f"[ПРИМЕЧАНИЕ] Страницы с нетипичным языком "
          f"({len(LANG_OUTLIER_PAGES)} шт.) будут отмечены в отчёте тональности.")


## 2) Загрузка языковой модели spaCy

spaCy использует предобученные модели для токенизации, лемматизации, POS-теггинга и NER.  
Модель выбирается **автоматически** по определённому выше языку.  

> **Важно:** Ноутбук использует модели версии **Large (lg)**, которые содержат
> полные словарные векторы. Это значительно улучшает качество тематизации (KMeans)
> и распознавания именованных сущностей (NER) по сравнению с версиями `sm`.

| Язык | Код | Модель |
|------|-----|--------|
| English | `en` | `en_core_web_lg` |
| Polish | `pl` | `pl_core_news_lg` |
| Russian | `ru` | `ru_core_news_lg` |
| Italian | `it` | `it_core_news_lg` |
| Finnish | `fi` | `fi_core_news_lg` |
| Icelandic | `is` | `spacy.blank("is")` — только базовая токенизация |

Если модель не установлена, ноутбук использует базовый токенизатор (`spacy.blank`) и сообщает команду для установки.

In [ ]:
import spacy

MODEL_BY_LANG = {
    "en": "en_core_web_lg",
    "pl": "pl_core_news_lg",
    "ru": "ru_core_news_lg",
    "it": "it_core_news_lg",
    "fi": "fi_core_news_lg",
}

# Hugging Face модели для исландского (нет в spaCy):
HF_IS_VECTORS = "mideind/IceBERT-base"
HF_IS_NER = "mideind/icelandic-ner-MIM-GOLD-22"


def _silence_hf_progress():
    """Тушит progress bars и предупреждения transformers/torch для чистого stdout."""
    import os
    os.environ["TRANSFORMERS_VERBOSITY"] = "error"
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        import transformers
        transformers.logging.set_verbosity_error()
        transformers.utils.logging.disable_progress_bar()
    except Exception:
        pass


def _build_icelandic_pipeline(blank_nlp):
    """Добавляет к spacy.blank('is') два HF-компонента: векторы (IceBERT) и NER.
    Если transformers/torch недоступны или сеть упала — возвращает blank без изменений.
    """
    try:
        _silence_hf_progress()
        from transformers import AutoTokenizer, AutoModel, pipeline
        import torch
        import numpy as np
        from spacy.language import Language
        from spacy.tokens import Doc
    except ImportError as _e:
        print(f"[ВНИМАНИЕ] transformers/torch недоступны: {_e}")
        print("  Пайплайн для исландского остаётся базовым (без NER и векторов).")
        return blank_nlp

    # --- Векторы через IceBERT-base ---
    try:
        print(f"  Загрузка эмбеддингов: {HF_IS_VECTORS} ...")
        _is_tok = AutoTokenizer.from_pretrained(HF_IS_VECTORS)
        _is_mdl = AutoModel.from_pretrained(HF_IS_VECTORS)
        _is_mdl.eval()

        def _is_embed(text):
            text = (text or "").strip()
            if not text:
                return np.zeros(_is_mdl.config.hidden_size, dtype=np.float32)
            with torch.no_grad():
                enc = _is_tok(text, return_tensors="pt", truncation=True, max_length=512)
                out = _is_mdl(**enc)
                mask = enc["attention_mask"].unsqueeze(-1).float()
                pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1)
            return pooled.squeeze(0).cpu().numpy().astype(np.float32)

        @Language.component("icebert_vectors")
        def _icebert_vectors_pipe(doc):
            vec = _is_embed(doc.text)
            doc.user_hooks["vector"] = lambda d: vec
            doc.user_hooks["has_vector"] = lambda d: True
            doc.user_hooks["vector_norm"] = lambda d: float((vec * vec).sum() ** 0.5)
            return doc

        if "icebert_vectors" not in blank_nlp.pipe_names:
            blank_nlp.add_pipe("icebert_vectors", last=True)
        print(f"  [OK] Эмбеддинги активны: размерность {_is_mdl.config.hidden_size}.")
    except Exception as _e:
        print(f"[ВНИМАНИЕ] Не удалось загрузить {HF_IS_VECTORS}: {_e}")
        print("  Тематизация будет пропущена.")

    # --- NER через icelandic-ner-MIM-GOLD-22 ---
    try:
        print(f"  Загрузка NER: {HF_IS_NER} ...")
        _is_ner = pipeline(
            "token-classification",
            model=HF_IS_NER,
            aggregation_strategy="simple",
        )

        @Language.component("icelandic_ner")
        def _icelandic_ner_pipe(doc):
            from spacy.tokens import Span
            text = doc.text or ""
            if not text.strip():
                return doc
            try:
                results = _is_ner(text[:5000]) or []
            except Exception:
                return doc
            spans = []
            for r in results:
                start_c = int(r.get("start", 0))
                end_c = int(r.get("end", 0))
                label = str(r.get("entity_group") or r.get("entity") or "MISC")
                if end_c <= start_c:
                    continue
                span = doc.char_span(start_c, end_c, label=label,
                                      alignment_mode="contract")
                if span is not None:
                    spans.append(span)
            try:
                doc.set_ents(spans)
            except Exception:
                pass
            return doc

        if "icelandic_ner" not in blank_nlp.pipe_names:
            blank_nlp.add_pipe("icelandic_ner", last=True)
        print("  [OK] Исландский NER активен.")
    except Exception as _e:
        print(f"[ВНИМАНИЕ] Не удалось загрузить {HF_IS_NER}: {_e}")
        print("  NER для исландского будет пропущен.")

    return blank_nlp


def load_nlp_model(lang):
    if lang == "is":
        print("[ИНФО] Для исландского spaCy не имеет полной модели — подключаем модели Hugging Face.")
        nlp_ = spacy.blank("is")
        if "sentencizer" not in nlp_.pipe_names:
            nlp_.add_pipe("sentencizer")
        nlp_ = _build_icelandic_pipeline(nlp_)
        return nlp_
    model_name = MODEL_BY_LANG.get(lang, "en_core_web_lg")
    try:
        nlp_ = spacy.load(model_name)
        print(f"[OK] Загружена модель spaCy: {model_name}")
        print(f"     Компоненты конвейера: {nlp_.pipe_names}")
        return nlp_
    except OSError:
        print(f"[ВНИМАНИЕ] Модель '{model_name}' не установлена.")
        print(f"  Установите: python -m spacy download {model_name}")
        print(f"  Доступные модели (lg):")
        print(f"    python -m spacy download en_core_web_lg")
        print(f"    python -m spacy download pl_core_news_lg")
        print(f"    python -m spacy download ru_core_news_lg")
        print(f"    python -m spacy download it_core_news_lg")
        print(f"    python -m spacy download fi_core_news_lg")
        valid_blanks = {"en", "pl", "ru", "it", "fi"}
        nlp_ = spacy.blank(lang if lang in valid_blanks else "en")
        if "sentencizer" not in nlp_.pipe_names:
            nlp_.add_pipe("sentencizer")
        return nlp_

print("--- Загрузка языковой модели ---")
nlp = load_nlp_model(LANG)
stop_words = nlp.Defaults.stop_words
print(f"\nСписок стоп-слов: {len(stop_words)} записей для {LANG_NAMES.get(LANG, LANG)}")


## Коррекция текста и нормализация аббревиатур

Автоматически исправляет типичные ошибки оформления во всех поддерживаемых языках.

**Для всех языков:**
- Удаление точных повторов слов (`the the`, `jest jest`, `что что` и т.п.)
- Схлопывание множественных пробелов

**Нормализация аббревиатур (польский `pl`):**
- `m.in`, `m. in`, `m.in,` → `m.in.`; `np.,` → `np.`; `tzw.,` → `tzw.`; `tzn.` — полный набор
- `tj.`, `itd.`, `itp.` — склейка разбитых аббревиатур

**Нормализация аббревиатур (русский `ru`):**
- `т. е.` → `т.е.`; `т. д.` → `т.д.`; `и т.д.` → единый вид; и другие

**Нормализация аббревиатур (английский `en`):**
- `e. g.` → `e.g.`; `e.g,` → `e.g.,`; `i. e.` → `i.e.`; `etc .` → `etc.`

> Цель нормализации: предотвратить ложное разбиение предложений токенизатором spaCy на аббревиатурах.


In [ ]:
# Коррекция типичных ошибок оформления (только regex, без морфологии).
# Применяется ко всем поддерживаемым языкам.

# ── Удаление паттернов CUSTOM_PATTERNS ───────────────────────────────────
if CUSTOM_PATTERNS:
    _n_custom = 0
    cleaned_custom = []
    for _ci, _ct in enumerate(corpus, 1):
        _fixed = _ct
        for _cp in CUSTOM_PATTERNS:
            _new = re.sub(_cp, "", _fixed)
            if _new != _fixed:
                _n_custom += 1
                _fixed = _new
        _fixed = re.sub(r"  +", " ", _fixed).strip()
        cleaned_custom.append(_fixed)
    corpus = cleaned_custom
    print(f"CUSTOM_PATTERNS: удалено совпадений в {_n_custom} документе(-ах).")

corpus_issues = []

import re

# ── Паттерны аббревиатур по языкам ────────────────────────────────
ABBREV_BY_LANG = {
    "pl": [
        # m.in. – między innymi (różne zapisy z brakiem lub nadmiarem kropek/spacji)
        (r"\bm\.\s*in\.?,?\b",        "m.in."),
        (r"\bmi\.in\.?\b",              "m.in."),
        (r"\bm\.\s+in\.",               "m.in."),
        # np. – na przykład
        (r"\bnp\.?,?\s",                 "np. "),
        (r"\bn\.\s*p\.\b",             "np."),
        # tzw. – tak zwany
        (r"\btzw\.?,?\s",                "tzw. "),
        (r"\bt\.\s*z\.\s*w\.\b",    "tzw."),
        # tzn. – to znaczy
        (r"\btzn\.?,?\s",                "tzn. "),
        (r"\bt\.\s*z\.\s*n\.\b",    "tzn."),
        # tj. – to jest
        (r"\bt\.\s*j\.\b",             "tj."),
        # itd. / itp.
        (r"\bi\.\s*t\.\s*d\.\b",    "itd."),
        (r"\bi\.\s*t\.\s*p\.\b",    "itp."),
        # ww. – wyżej wymieniony
        (r"\bww\.?,?\s",                 "ww. "),
        # jw. – jak wyżej
        (r"\bjw\.?,?\s",                 "jw. "),
        # dr / prof / mgr / inż – tytuły
        (r"\bdr\s+hab\.?\s",            "dr hab. "),
        (r"\bdr\.?\s",                   "dr "),
        (r"\bprof\.?\s",                 "prof. "),
        (r"\bmgr\.?\s",                  "mgr "),
        (r"\bin\u017c\.?\s",            "in\u017c. "),
        # ul. / al. / pl. – adresy
        (r"\bul\.?,?\s",                 "ul. "),
        (r"\bal\.?,?\s",                 "al. "),
        (r"\bpl\.?,?\s",                 "pl. "),
        # r. – rok; nr – numer; str. – strona
        (r"\br\.?,?\s",                  "r. "),
        (r"\bnr\s*\.?,?\s",             "nr "),
        (r"\bstr\.?,?\s",                "str. "),
    ],
    "ru": [
        (r"\b\u0442\.\s+\u0435\.",             "\u0442.\u0435."),    # т. е. → т.е.
        (r"\b\u0442\.\s+\u0434\.",             "\u0442.\u0434."),    # т. д. → т.д.
        (r"\b\u0442\.\s+\u043f\.",             "\u0442.\u043f."),    # т. п. → т.п.
        (r"\b\u0442\.\s+\u043a\.",             "\u0442.\u043a."),    # т. к. → т.к.
        (r"\b\u0442\.\s+\u043d\.",             "\u0442.\u043d."),    # т. н. → т.н.
        (r"\b\u0438\.\s+\u043e\.",             "\u0438.\u043e."),    # и. о. → и.о.
        (r"\b\u0441\.\s+\u0433\.",             "\u0441.\u0433."),    # с. г. → с.г.
        (r"\b\u043d\.\s+\u044d\.",             "\u043d.\u044d."),    # н. э. → н.э.
        (r"\b\u0432\.\s+\u0442\.\s+\u0447\.", "\u0432.\u0442.\u0447."),  # в. т. ч. → в.т.ч.
        # и т.д. / и т.п. – с пробелом внутри
        (r"\u0438\s+\u0442\.\s*\u0434\.",     "\u0438 \u0442.\u0434."),   # и т.д.
        (r"\u0438\s+\u0442\.\s*\u043f\.",     "\u0438 \u0442.\u043f."),   # и т.п.
        # проф. / акад. / др. – титулы
        (r"\b\u043f\u0440\u043e\u0444\.?,?\s", "\u043f\u0440\u043e\u0444. "),  # проф.
        (r"\b\u0430\u043a\u0430\u0434\.?,?\s", "\u0430\u043a\u0430\u0434. "), # акад.
        (r"\b\u0434\u043e\u0446\.?,?\s",        "\u0434\u043e\u0446. "),         # доц.
        # ул. / пр. / пер. – адреса
        (r"\b\u0443\u043b\.?,?\s",               "\u0443\u043b. "),    # ул.
        (r"\b\u043f\u0440\.?,?\s",               "\u043f\u0440. "),    # пр.
        (r"\b\u043f\u0435\u0440\.?,?\s",        "\u043f\u0435\u0440. "),  # пер.
    ],
    "en": [
        (r"\be\.\s+g\.",       "e.g."),    # e. g. → e.g.
        (r"\be\.g,",             "e.g.,"),   # e.g, → e.g.,
        (r"\bi\.\s+e\.",       "i.e."),    # i. e. → i.e.
        (r"\bi\.e,",             "i.e.,"),   # i.e, → i.e.,
        (r"\bU\.\s+S\.",       "U.S."),    # U. S. → U.S.
        (r"\bU\.\s+K\.",       "U.K."),    # U. K. → U.K.
        (r"\betc\s+\.",         "etc."),    # etc . → etc.
        (r"\bvs\.\s+",          "vs. "),    # vs.  → vs. (normalizacja)
        (r"\bDr\.\s+",          "Dr. "),    # Dr.  (tytuł)
        (r"\bMr\.\s{2,}",       "Mr. "),    # Mr.  (tytuł)
        (r"\bMrs\.\s{2,}",      "Mrs. "),   # Mrs. (tytuł)
        (r"\bProf\.\s+",        "Prof. "),  # Prof.
        (r"\bSt\.\s+",          "St. "),    # St. (Saint / Street)
        (r"\bFig\.\s+",         "Fig. "),   # Fig. (Figure)
        (r"\bNo\.\s+",          "No. "),    # No. (Number)
        (r"\bp\.\s+",           "p. "),     # p. (page)
        (r"\bpp\.\s+",          "pp. "),    # pp. (pages)
    ],
    "it": [
        (r"\bad\s+es\.?,?\b",  "ad es."),  # ad esempio
        (r"\becc\.?,?\b",       "ecc."),    # eccetera
        (r"\bdott\.?,?\s",      "dott. "),  # dottore
        (r"\bprof\.?,?\s",      "prof. "),  # professore
        (r"\bpagg?\.?,?\s",     "pag. "),   # pagina
        (r"\bsig\.?,?\s",       "sig. "),   # signore
        (r"\bsig\.?ra\.?,?\s", "sig.ra "), # signora
        (r"\bart\.?,?\s",       "art. "),   # articolo
        (r"\bcap\.?,?\s",       "cap. "),   # capitolo
        (r"\bn\.\s*ro?\.?\b", "n."),      # numero
        (r"\bcfr\.?,?\s",       "cfr. "),   # confronta
        (r"\bvol\.?,?\s",       "vol. "),   # volume
        (r"\bpag\.?,?\s",       "pag. "),   # pagina (alias)
    ],
    "fi": [
        (r"\besim\.?,?\s",      "esim. "),  # esimerkiksi
        (r"\bjne\.?,?\b",       "jne."),    # ja niin edelleen
        (r"\bym\.?,?\b",        "ym."),     # ynnä muuta
        (r"\bns\.?,?\s",        "ns. "),    # niin sanottu
        (r"\btms\.?,?\b",       "tms."),    # tai muuta sellaista
        (r"\bko\.?,?\s",        "ko. "),    # kyseinen / kyseessä oleva
        (r"\bpo\.?,?\s",        "po. "),    # pohjoiseen / pohjoinen
        (r"\bvt\.?,?\s",        "vt. "),    # virkaatekevä
        (r"\bprof\.?,?\s",      "prof. "),  # professori
        (r"\bdr\.?,?\s",        "dr. "),    # tohtori
        (r"\bos\.?,?\s",        "os. "),    # osasto
        (r"\bv\.?,?\s",         "v. "),     # vuosi / versus
    ],
    "is": [
        (r"\bt\.\s*d\.?,?\b",             "t.d."),      # til dæmis
        (r"\b\u00fe\.\s*e\.?,?\b",        "\u00fe.e."),  # þ.e. – það er
        (r"\bm\.\s*a\.?,?\b",             "m.a."),      # meðal annars
        (r"\bu\.\s*\u00fe\.\s*b\.?,?\b", "u.\u00fe.b."),  # u.þ.b. – um það bil
        (r"\bo\.\s*s\.\s*frv\.?,?\b",  "o.s.frv."),  # og svo framvegis
        (r"\bdr\.?,?\s",                    "dr. "),       # doktor
        (r"\bprof\.?,?\s",                  "prof. "),     # prófessor
        (r"\bbls\.?,?\s",                   "bls. "),      # blaðsíða (strona)
        (r"\bskv\.?,?\s",                   "skv. "),      # samkvæmt (zgodnie z)
        (r"\bfh\.?,?\s",                    "fh. "),       # fyrir hönd (w imieniu)
    ],
}
abbrev_patterns = ABBREV_BY_LANG.get(LANG, [])

corrected = []
for i, original in enumerate(corpus, 1):
    fixed = original
    # 1. Языко-специфичные аббревиатуры
    for pattern, replacement in abbrev_patterns:
        new_text = re.sub(pattern, replacement, fixed)
        if new_text != fixed:
            corpus_issues.append((i, "исправлено",
                f"аббревиатура: шаблон '{pattern}' → '{replacement}'"))
            fixed = new_text
    # 2. Повтор слова (все языки, без учёта регистра)
    def _fix_repeat(m):
        w = m.group(1)
        corpus_issues.append((i, "исправлено",
            f"повтор слова: '{w} {w}' → '{w}'"))
        return w
    fixed = re.sub(r"\b(\w{2,})\s+\1\b", _fix_repeat, fixed,
                   flags=re.IGNORECASE)
    # 3. Множественные пробелы
    fixed = re.sub(r"  +", " ", fixed)
    corrected.append(fixed)

corpus = corrected

if corpus_issues:
    _lang_label = LANG_NAMES.get(LANG, LANG)
    print(f"Коррекция текста ({_lang_label}): "
          f"исправлено {len(corpus_issues)} проблем(ы).")
    for doc_n, kind, desc in corpus_issues[:20]:
        print(f"  [Исправлено] Документ {doc_n}: {desc}")
    if len(corpus_issues) > 20:
        print(f"  ... и ещё {len(corpus_issues)-20} исправлений")
else:
    _lang_label = LANG_NAMES.get(LANG, LANG)
    print(f"Коррекция текста ({_lang_label}): типичных ошибок не обнаружено.")


## 3) Токенизация

**Токенизация** — разбиение текста на элементарные единицы: слова и знаки препинания.  
spaCy делает это с учётом языка: правильно обрабатывает апострофы, дефисы, аббревиатуры.  

### 3.1 Токенизация на слова
Каждый документ превращается в список токенов. Пробелы исключаются.

### 3.2 Токенизация на предложения
spaCy также сегментирует текст на предложения — полезно для анализа структуры и для сентимент-анализа по предложениям.

In [ ]:
print("--- Токенизация ---")
docs = [nlp(t) for t in corpus]

word_tokens = [[tok.text for tok in doc if not tok.is_space] for doc in docs]
print("Словесные токены:")
for i, tokens in enumerate(word_tokens, 1):
    print(f"  Документ {i}: всего токенов — {len(tokens)}")
    print(f"    Первые 20: {tokens[:20]}")

sent_tokens = [[s.text.strip() for s in doc.sents if s.text.strip()] for doc in docs]
print("\nПредложения:")
MAX_SENT_PREVIEW = 3
for i, sents in enumerate(sent_tokens, 1):
    print(f"  Документ {i}: предложений — {len(sents)}")
    for j, s in enumerate(sents[:MAX_SENT_PREVIEW], 1):
        _s = s.replace("\n", " ")[:100]
        print(f"    [{j}] {_s}")
    if len(sents) > MAX_SENT_PREVIEW:
        print(f"    ... и ещё {len(sents) - MAX_SENT_PREVIEW} предложений")


## 4) Стоп-слова

**Стоп-слова** — служебные слова (артикли, предлоги, местоимения), которые мало несут смысла.  
Список берётся прямо из загруженной модели spaCy — он уже оптимизирован под язык корпуса.  
После фильтрации остаются только семантически насыщенные слова.

In [ ]:
print("--- Фильтрация стоп-слов ---")
print(f"Язык: {LANG_NAMES.get(LANG, LANG)} | Стоп-слов в списке: {len(stop_words)}")
print(f"Выборка (первые 20 по алфавиту): {sorted(list(stop_words))[:20]}")

filtered_tokens = [
    [tok.text for tok in doc if tok.is_alpha and tok.text.lower() not in stop_words]
    for doc in docs
]
print()
for i, tokens in enumerate(filtered_tokens, 1):
    print(f"  Документ {i}: содержательных слов после фильтрации — {len(tokens)}")
    print(f"    Выборка: {tokens[:20]}")


## 5) Лемматизация

**Лемма** — начальная (словарная) форма слова. Лемматизация приводит все формы к единому виду:  
«бежит», «бежали», «бегут» → «бежать»; «documents», «document» → «document».  

В отличие от стемминга, лемматизация использует словарь и грамматические правила — результат всегда читаемое слово.  
В spaCy лемма доступна через `token.lemma_`.

In [ ]:
print("--- Лемматизация ---")
all_lemmas = [
    [t.lemma_.lower() for t in doc if t.is_alpha and t.text.lower() not in stop_words]
    for doc in docs
]
doc0 = docs[0]
content_toks = [t for t in doc0 if t.is_alpha and t.text.lower() not in stop_words]
pairs = [(t.text, t.lemma_.lower()) for t in content_toks[:30]]
print("Документ 1 — пары исходное слово → лемма (первые 30):")
_h1, _h2, _sep = "Исходное", "Лемма", "-" * 22
print(f"  {_h1:<22} {_h2}")
print(f"  {_sep} -----")
for orig, lemma in pairs:
    changed = "  <-- изменено" if orig.lower() != lemma else ""
    print(f"  {orig:<22} {lemma}{changed}")
print()
for i, lemmas in enumerate(all_lemmas, 1):
    print(f"  Документ {i}: лемм — {len(lemmas)}, уникальных — {len(set(lemmas))}")


## 6) Части речи (POS-теггинг)

**POS-теггинг** присваивает каждому токену грамматическую роль: существительное, глагол, прилагательное и т.д.  
spaCy возвращает два уровня:
- `pos_` — универсальный тег (NOUN, VERB, ADJ, ADV, PROPN …)
- `tag_` — детальный тег, специфичный для языка

Распределение POS позволяет понять стиль текста: в научных текстах много существительных, в художественных — глаголов и прилагательных.

In [ ]:
print("--- Разметка частей речи (POS) ---")
print("Теги: NOUN — сущ., VERB — гл., ADJ — прил., ADV — нар., PROPN — имя собств., ...")
for i, doc in enumerate(docs, 1):
    pos_pairs = [(t.text, t.pos_) for t in doc if t.is_alpha]
    pos_counts = Counter(pos for _, pos in pos_pairs)
    print(f"\nДокумент {i} — распределение частей речи:")
    for pos, cnt in sorted(pos_counts.items(), key=lambda x: -x[1]):
        bar = "#" * min(cnt, 40)
        print(f"  {pos:8s}: {cnt:4d}  {bar}")
    notable = [(t.text, t.pos_, t.tag_) for t in doc
               if t.is_alpha and t.text.lower() not in stop_words][:10]
    print("  Примеры тегированных слов (слово | POS | детальный тег):")
    for txt, pos, tag in notable:
        print(f"    {txt!r:<22} | {pos:<10} | {tag}")


## 7) Именованные сущности (NER)

**NER** (Named Entity Recognition) находит имена, организации, места, даты, суммы денег и другие упомянутые объекты.  
spaCy возвращает список `doc.ents`, где у каждой сущности есть текст и метка.  

Типичные метки: PERSON, ORG, GPE (город/страна), DATE, MONEY, PRODUCT.

In [ ]:
print("--- Именованные сущности (NER) ---")
all_ents = []
for i, doc in enumerate(docs, 1):
    ents = [(ent.text, ent.label_) for ent in doc.ents]
    all_ents.extend(ents)
    if ents:
        print(f"Документ {i}: найдено сущностей — {len(ents)}")
        for txt, label in ents:
            print(f"  [{label}] {txt!r}")
    else:
        print(f"Документ {i}: именованных сущностей не обнаружено.")
    print()
if all_ents:
    print("Типы сущностей по всему корпусу:")
    for etype, cnt in Counter(label for _, label in all_ents).most_common():
        print(f"  {etype:<12}: {cnt}")
else:
    print("Именованных сущностей не найдено ни в одном документе.")


## 8) Мешок слов (CountVectorizer)

**CountVectorizer** превращает тексты в числовые векторы, считая, сколько раз каждое слово встречается в документе.  
Результат — матрица «документ × слово», где каждая строка — один документ, а каждый столбец — одно слово.  

### 8.1 Униграммы (отдельные слова)
Базовое представление: каждое слово — отдельный признак.

### 8.2 Биграммы (пары слов)
Добавляет устойчивые выражения: «machine learning», «natural language», «named entity».

In [ ]:
print("--- Мешок слов (CountVectorizer) ---")
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

sklearn_stop = list(stop_words) if stop_words else "english"

vec_uni = CountVectorizer(stop_words=sklearn_stop, min_df=1)
X_uni = vec_uni.fit_transform(corpus)
vocab_uni = vec_uni.get_feature_names_out()
print(f"Словарь униграмм: {len(vocab_uni)} уникальных терминов")
print(f"Матрица документ×слово: {X_uni.shape}")
df_uni = pd.DataFrame(X_uni.toarray(), columns=vocab_uni,
                      index=[f"Документ {i+1}" for i in range(len(corpus))])
print("\nТоп-10 наиболее частых слов по всему корпусу:")
for term, freq in df_uni.sum().nlargest(10).items():
    _bar = "#" * int(freq)
    print(f"  {term!r:<25}: {int(freq):3d}  {_bar}")

vec_bi = CountVectorizer(ngram_range=(2,2), stop_words=sklearn_stop, max_features=50, min_df=1)
X_bi = vec_bi.fit_transform(corpus)
vocab_bi = vec_bi.get_feature_names_out()
print(f"\nСловарь биграмм (топ-50): {len(vocab_bi)} уникальных биграмм")
top5_bi = pd.DataFrame(X_bi.toarray(), columns=vocab_bi).sum().nlargest(5)
if not top5_bi.empty:
    print("Топ-5 наиболее частых биграмм:")
    for rank_b, (term, freq) in enumerate(top5_bi.items(), 1):
        print(f"  {rank_b}. {term!r}: {int(freq)}")
else:
    print("Биграммы не найдены (корпус слишком мал).")

vec_tri = CountVectorizer(ngram_range=(3,3), stop_words=sklearn_stop, max_features=50, min_df=1)
X_tri = vec_tri.fit_transform(corpus)
vocab_tri = vec_tri.get_feature_names_out()
print(f"\nСловарь триграмм (топ-50): {len(vocab_tri)} уникальных триграмм")
top5_tri = pd.DataFrame(X_tri.toarray(), columns=vocab_tri).sum().nlargest(5)
if not top5_tri.empty:
    print("Топ-5 наиболее частых триграмм (трёхсловных выражений):")
    for rank_t, (term, freq) in enumerate(top5_tri.items(), 1):
        print(f"  {rank_t}. {term!r}: {int(freq)}")
else:
    print("Триграммы не найдены (корпус слишком мал).")


## 9) TF-IDF и автоматический поиск

**TF-IDF** (Term Frequency — Inverse Document Frequency) взвешивает слова: частые в одном документе, но редкие в остальных, получают высокий вес.  
Это делает поиск умнее, чем простой подсчёт слов.

**Косинусное сходство** измеряет угол между векторами двух текстов:  
- 1.0 = полностью совпадают  
- 0.0 = не имеют ни одного общего слова

**Автоматический запрос:** ноутбук строит запрос из слов с наибольшим средним весом TF-IDF по корпусу.  
Такой запрос гарантированно даёт косинусное сходство > 0 (никогда не возвращает −1).

In [ ]:
print("--- TF-IDF-поиск с автоматическим запросом ---")
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

random.seed(42)
tfidf = TfidfVectorizer(stop_words=sklearn_stop, min_df=1)
tfidf_matrix = tfidf.fit_transform(corpus)
feature_names = tfidf.get_feature_names_out()
print(f"Матрица TF-IDF: {tfidf_matrix.shape} (документов × признаков)")
print(f"Общий словарь: {len(feature_names)} терминов")

mean_scores = tfidf_matrix.toarray().mean(axis=0)
top_indices = mean_scores.argsort()[::-1]
candidates = [feature_names[i] for i in top_indices
              if feature_names[i].isalpha() and len(feature_names[i]) > 3][:40]
if not candidates:
    candidates = [feature_names[i] for i in top_indices[:10]]
n_terms = random.randint(2, min(4, len(candidates)))
query_terms = random.sample(candidates, n_terms)
AUTO_QUERY = " ".join(query_terms)
print(f"\nАвтозапрос: \"{AUTO_QUERY}\"")
print("(Из топ-терминов TF-IDF — схожесть всегда > 0)")

q_vec = tfidf.transform([AUTO_QUERY])
cosine_scores = linear_kernel(q_vec, tfidf_matrix).flatten()
ranked_order = cosine_scores.argsort()[::-1]
_c1, _c2, _c3 = "Ранг", "Документ", "Схожесть"
print(f"\nРанжирование по запросу \"{AUTO_QUERY}\":\n")
print(f"  {_c1:<6} {_c2:<12} {_c3:<16} Фрагмент")
print("  " + "-" * 72)
for rank, idx2 in enumerate(ranked_order, 1):
    score = cosine_scores[idx2]
    preview = corpus[idx2].strip().replace("\n", " ")[:60]
    print(f"  {rank:<6} Документ {idx2+1:<5} {score:<16.4f} {preview}...")
best_idx = ranked_order[0]
print(f"\nНаиболее релевантный: Документ {best_idx+1} (косинус = {cosine_scores[best_idx]:.4f})")


## 10) Сентимент-анализ (тональность)

Используем многоязычную модель `cardiffnlp/twitter-xlm-roberta-base-sentiment`.  
Она поддерживает более 100 языков (в т.ч. **польский**, **русский**, **финский**, **исландский**, **итальянский**, **английский**)  
и возвращает одну из трёх меток: **Негативно**, **Нейтрально**, **Позитивно**.

**Первый запуск** скачивает веса модели (~1.1 ГБ). Все последующие используют локальный кэш.  

Если загрузка не удаётся (нет интернета), ноутбук автоматически переключится на упрощённую английскую модель.

In [ ]:
import os
import warnings

# Pełne wyciszenie Hugging Face i Transformers
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")

import transformers
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()  # blokuje pasek "Loading weights"


print("--- Анализ тональности ---")
print("Модель: cardiffnlp/twitter-xlm-roberta-base-sentiment")
print("Поддерживает более 100 языков (pl, ru, fi, is, it, en и др.)")
print("Шкала: Негативно | Нейтрально | Позитивно")
print()
from transformers import pipeline as hf_pipeline
from collections import Counter

# Маппинг меток модели на русские названия тональности.
# Поддерживает оба формата: новый ('negative'/'neutral'/'positive')
# и старый ('LABEL_0'/'LABEL_1'/'LABEL_2').
LABEL_MAP = {
    "negative": "Негативно",
    "neutral":  "Нейтрально",
    "positive": "Позитивно",
    "label_0":  "Негативно",
    "label_1":  "Нейтрально",
    "label_2":  "Позитивно",
    "n/a":      "Н/Д (модель недоступна)",
    "error":    "Ошибка анализа",
}

# Для совместимости с ячейкой итогового отчёта
STAR_MAP = LABEL_MAP

_outliers = LANG_OUTLIER_PAGES if "LANG_OUTLIER_PAGES" in dir() else set()

# Порог уверенности: фрагменты ниже этого значения исключаются из агрегата.
# В 3-классовой модели 33% = случайный угад; 40% ≈ минимальный надёжный сигнал.
CONF_THRESHOLD = 0.40

sentiment_clf = None
try:
    sentiment_clf = hf_pipeline(
        "sentiment-analysis",
        model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
        truncation=True, max_length=512)
    print("[OK] Многоязычная модель тональности загружена.")
except Exception as _exc1:
    print(f"[ВНИМАНИЕ] Многоязычная модель недоступна: {_exc1}")
    print("  Попытка загрузить стандартную английскую модель...")
    try:
        sentiment_clf = hf_pipeline(
            "sentiment-analysis",
            truncation=True, max_length=512)
        print("[OK] Английская модель тональности загружена.")
    except Exception as _exc2:
        print(f"[ВНИМАНИЕ] Обе модели тональности недоступны: {_exc2}")
        print("  Анализ тональности будет пропущен.")

sentiment_results = []
print()
print("-" * 60)
if sentiment_clf is None:
    print("[ВНИМАНИЕ] Анализ тональности пропущен (нет доступной модели).")
    sentiment_results = [("n/a", 0.0)] * len(corpus)
else:
    for i, text in enumerate(corpus, 1):
        try:
            result = sentiment_clf(text[:512])[0]
            label_raw = result["label"]
            score = result["score"]
            # Normalizujemy etykietę do małych liter dla pewnego mapowania
            label_key = label_raw.lower()
            readable = LABEL_MAP.get(label_key, label_raw)
            # Przechowujemy znormalizowany klucz, by agregat działał poprawnie
            sentiment_results.append((label_key, score))
            is_outlier = (i - 1) in _outliers
            is_low_conf = score < CONF_THRESHOLD
            if is_outlier and is_low_conf:
                flag = ("  [!?] иноязычный + низкая уверенность "
                        "— исключён из агрегата")
            elif is_outlier:
                flag = ("  [!] иноязычный фрагмент "
                        "— результат может быть нерепрезентативным")
            elif is_low_conf:
                _pct = f"{score:.2%}"
                _thr = f"{CONF_THRESHOLD:.0%}"
                flag = f"  [?] уверенность {_pct} < порога {_thr} — исключён из агрегата"
            else:
                flag = ""
            print(f"Документ {i}: {readable} ({score:.2%}){flag}")
        except Exception as exc:
            print(f"Документ {i}: анализ не выполнен — {exc}")
            sentiment_results.append(("error", 0.0))

# ── Агрегат: исключаем иноязычные и низкоуверенные фрагменты ─────────
if sentiment_results:
    reliable = [
        (l, s) for idx, (l, s) in enumerate(sentiment_results)
        if idx not in _outliers and s >= CONF_THRESHOLD
    ]
    excluded = len(sentiment_results) - len(reliable)
    print()
    if excluded:
        _th = f"{CONF_THRESHOLD:.0%}"
        print(f"Исключено из агрегата: {excluded} стр. "
              f"(иноязычные или уверенность < {_th})")
    _use = reliable if reliable else sentiment_results
    _n_use = len(_use)
    _n_tot = len(sentiment_results)
    if excluded:
        print(f"Агрегат — надёжные страницы ({_n_use}/{_n_tot}):")
    else:
        print(f"Агрегат по всем {_n_use} страницам:")
    dist = Counter(LABEL_MAP.get(l, l) for l, _ in _use)
    n_neg  = dist.get("Негативно", 0)
    n_neut = dist.get("Нейтрально", 0)
    n_pos  = dist.get("Позитивно", 0)
    print(f"  Позитивно  : {n_pos:3d} фрагм. ({n_pos/_n_use*100:.0f}%)")
    print(f"  Нейтрально : {n_neut:3d} фрагм. ({n_neut/_n_use*100:.0f}%)")
    print(f"  Негативно  : {n_neg:3d} фрагм. ({n_neg/_n_use*100:.0f}%)")


## Структура текста: абзацы и предложения

Для извлечения тезисов и тематизации нам нужна более тонкая разбивка: объединяем весь корпус в единый текст, делим spaCy на предложения, затем группируем по 3–6 предложений в «абзацы». Эта структура используется в последующих шагах.

In [ ]:
print("--- Структура текста: абзацы ---")
import re, numpy as np
import pandas as pd

# Объединяем все страницы/куски корпуса в один поток
full_text = " ".join(c.replace("\n", " ") for c in corpus)
full_text = re.sub(r"\s+", " ", full_text).strip()
print(f"Общий объём текста: {len(full_text)} символов")

# Разбиваем на предложения (spaCy)
full_doc = nlp(full_text)
all_sents = [s.text.strip() for s in full_doc.sents if len(s.text.strip()) > 10]
print(f"Всего предложений: {len(all_sents)}")

# Группируем в абзацы по 3-6 предложений
PARA_MIN, PARA_MAX = 3, 6
paragraphs, buf = [], []
for sent in all_sents:
    buf.append(sent)
    if len(buf) >= PARA_MAX:
        paragraphs.append(" ".join(buf))
        buf = []
if buf:
    paragraphs.append(" ".join(buf))
print(f"Сформировано абзацев: {len(paragraphs)} (по {PARA_MIN}–{PARA_MAX} предложений)")

df_sent = pd.DataFrame({"sent_id": range(1, len(all_sents)+1), "sentence": all_sents})
df_para = pd.DataFrame({"para_id": range(1, len(paragraphs)+1), "paragraph": paragraphs})

# Языково-специфичный паттерн токенов для CountVectorizer / TF-IDF
TOKEN_PAT = {
    "pl": r"(?u)\b[a-z\u0105\u0107\u0119\u0142\u0144\u00f3\u015b\u017a\u017c]{2,}\b",
    "ru": r"(?u)\b[\u0430-\u044f\u0451]{2,}\b",
    "en": r"(?u)\b[a-z]{2,}\b",
}.get(LANG, r"(?u)\b\S{2,}\b")

# Лемматизируем абзацы (понадобится для TF-IDF)
def to_lemma_str(text):
    out = []
    for t in nlp(text):
        if not t.is_alpha or t.is_stop or t.text.lower() in stop_words:
            continue
        out.append((t.lemma_ or t.text).lower())
    return " ".join(out)

para_lemmas = [to_lemma_str(p) for p in paragraphs]
print("Лемматизация абзацев завершена.")


## Ключевые слова (TF-IDF)

TF-IDF по абзацам выявляет термины, характерные для конкретных фрагментов, но редкие в целом по тексту. Здесь мы получаем рейтинговый список ключевых терминов и фраз (от 1 до 3 слов), отсортированных по средневзвешенному весу TF-IDF.

In [ ]:
print("--- Ключевые слова (TF-IDF) ---")
from sklearn.feature_extraction.text import TfidfVectorizer

kw_tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words=list(stop_words) if stop_words else "english",
    token_pattern=TOKEN_PAT,
    ngram_range=(1, 3),
    min_df=2
)
try:
    Xkw = kw_tfidf.fit_transform(para_lemmas)
    kw_terms  = kw_tfidf.get_feature_names_out()
    kw_scores = Xkw.mean(axis=0).A1
    df_keywords_tfidf = (
        pd.DataFrame({"term": kw_terms, "score": kw_scores})
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )
    print(f"Уникальных терминов: {len(kw_terms)}")
    print()
    _h1, _h2, _sep = "Термин", "Вес TF-IDF", "-" * 38
    print(f"  {_h1:<38} {_h2}")
    print(f"  {_sep} ----------")
    for _, row in df_keywords_tfidf.head(25).iterrows():
        _term, _sc = row["term"], row["score"]
        print(f"  {_term:<38} {_sc:.4f}")
except Exception as kw_err:
    print(f"[ВНИМАНИЕ] Не удалось извлечь ключевые слова: {kw_err}")
    print("  Возможная причина: слишком мало текста или все слова в стоп-листе.")
    df_keywords_tfidf = pd.DataFrame(columns=["term", "score"])


## Тезисы (экстрактивное саммари)

Для каждого абзаца выбирается одно **наиболее информативное предложение** (с максимальной суммой TF-IDF весов). Результат — компактный список тезисов, сохраняемый в файл `тезисы.txt` в формате, удобном для экранных читалок.

Параметр `N_THESES` задаёт максимальное число тезисов в файле.

In [ ]:
print("--- Тезисы ---")
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

N_THESES = 15  # максимальное число тезисов
THESES_FILE = str(PROJECT_DIR / "тезисы.txt")

# TF-IDF по предложениям
v_sent = TfidfVectorizer(
    lowercase=True,
    stop_words=list(stop_words) if stop_words else "english",
    token_pattern=TOKEN_PAT,
    ngram_range=(1, 2), min_df=1
)
try:
    S_sent = v_sent.fit_transform(df_sent["sentence"].tolist())
    sent_scores = np.asarray(S_sent.sum(axis=1)).ravel()
except Exception:
    sent_scores = np.ones(len(df_sent))

# Для каждого абзаца: найти лучшее предложение
theses_rows = []
sent_cursor = 0
for pid, para in enumerate(paragraphs, 1):
    para_sents = [s.text.strip() for s in nlp(para).sents if len(s.text.strip()) > 10]
    n = len(para_sents)
    if n == 0:
        continue
    seg = sent_scores[sent_cursor:sent_cursor + n]
    best_i = int(seg.argmax())
    best_sent = para_sents[best_i]
    if len(best_sent) >= 40:
        theses_rows.append({"para_id": pid, "sentence": best_sent,
                             "score": float(seg[best_i])})
    sent_cursor += n

df_theses = pd.DataFrame(theses_rows)
print(f"Сформировано тезисов: {len(df_theses)}")
print(f"Показываем первые {min(N_THESES, len(df_theses))}:")
print()
for _, row in df_theses.head(N_THESES).iterrows():
    _pid = int(row["para_id"])
    _sc  = row["score"]
    preview = row["sentence"][:150].replace("\n", " ")
    print(f"  Абзац {_pid:3d} (оценка {_sc:.3f}): {preview}")

# Сохраняем
theses_lines = ["- " + str(row["sentence"]) for _, row in df_theses.iterrows()]
theses_text = "\n".join(theses_lines)
with open(THESES_FILE, "w", encoding="utf-8") as f:
    f.write(theses_text)
print(f"\nТезисы сохранены: {THESES_FILE} ({len(df_theses)} тезисов)")


## Тематизация (KMeans)

Каждый абзац представляется вектором spaCy, затем кластеризуется алгоритмом KMeans. Для каждой темы выводятся ключевые слова и пример фрагмента.  

**Требование:** модель spaCy должна поддерживать векторы слов (например, `*_lg`). При использовании моделей без векторов тематизация пропускается с соответствующим сообщением.

In [ ]:
print("--- Тематизация ---")
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Проверяем наличие векторов в модели
_probe = nlp("test" if LANG == "en" else ("польский" if LANG == "ru" else "tekst"))
has_vectors = np.any(_probe.vector != 0) and _probe.has_vector

df_topics = pd.DataFrame()
topic_keywords = {}

if not has_vectors:
    _lg = MODEL_BY_LANG.get(LANG, "en_core_web_lg")
    print("[ВНИМАНИЕ] Модель spaCy не содержит векторов слов.")
    print(f"  Для тематизации установите: python -m spacy download {_lg}")
    print("  Тематизация пропущена.")
elif len(paragraphs) < 4:
    print("[ВНИМАНИЕ] Слишком мало абзацев для тематизации (нужно >= 4).")
else:
    vecs = []
    for para in paragraphs:
        v = nlp(para).vector
        vecs.append(v / (np.linalg.norm(v) + 1e-12))
    X_vec = np.vstack(vecs)
    N_TOPICS = min(6, max(2, len(paragraphs) // 4))
    kmeans = KMeans(n_clusters=N_TOPICS, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X_vec)
    df_topics = df_para.copy()
    df_topics["topic"] = labels
    print(f"Обнаружено тем: {N_TOPICS}  |  Абзацев: {len(paragraphs)}")
    print()
    # TF-IDF ключевые слова на тему
    for t in sorted(df_topics["topic"].unique()):
        subset = df_topics[df_topics["topic"] == t]["paragraph"].tolist()
        n_para = len(subset)
        try:
            vt = TfidfVectorizer(
                lowercase=True,
                stop_words=list(stop_words) if stop_words else "english",
                token_pattern=TOKEN_PAT,
                ngram_range=(1, 2), min_df=1)
            Xt = vt.fit_transform(subset)
            tt = vt.get_feature_names_out()
            st = Xt.mean(axis=0).A1
            top_t = pd.DataFrame({"term": tt, "s": st}).sort_values("s", ascending=False).head(8)["term"].tolist()
        except Exception:
            top_t = []
        topic_keywords[int(t)] = top_t
        kw_str = ", ".join(top_t) or "(нет данных)"
        print(f"Тема {t} ({n_para} абзацев):")
        print(f"  Ключевые слова: {kw_str}")
        ex = subset[0][:200].replace("\n", " ")
        print(f"  Пример: {ex}...")
        print()


## Экспорт результатов

Сохраняем все таблицы в папку `export_results/` в формате CSV (UTF-8 BOM, открывается в Excel) и JSON (ключевые слова тем).  

Файл `тезисы.txt` уже сохранён на предыдущем шаге.

In [ ]:
print("--- Экспорт результатов ---")
import json

out_dir = PROJECT_DIR  # каталог уже создан в cell_corpus
exported = []

# Предложения
df_sent.to_csv(out_dir / "sentences.csv", index=False, encoding="utf-8-sig")
exported.append("sentences.csv")

# Абзацы
df_para.to_csv(out_dir / "paragraphs.csv", index=False, encoding="utf-8-sig")
exported.append("paragraphs.csv")

# Тезисы CSV + TXT
if len(df_theses) > 0:
    df_theses.to_csv(out_dir / "theses.csv", index=False, encoding="utf-8-sig")
    (out_dir / "тезисы.txt").write_text(
        "\n".join(f"- {r['sentence']}" for _, r in df_theses.iterrows()),
        encoding="utf-8")
    exported += ["theses.csv", "тезисы.txt"]

# Ключевые слова TF-IDF
if len(df_keywords_tfidf) > 0:
    df_keywords_tfidf.to_csv(out_dir / "keywords_tfidf.csv", index=False, encoding="utf-8-sig")
    exported.append("keywords_tfidf.csv")

# Темы
if len(df_topics) > 0:
    df_topics.to_csv(out_dir / "paragraphs_with_topics.csv", index=False, encoding="utf-8-sig")
    exported.append("paragraphs_with_topics.csv")
    (out_dir / "topic_keywords.json").write_text(
        json.dumps(topic_keywords, ensure_ascii=False, indent=2), encoding="utf-8")
    exported.append("topic_keywords.json")

# Именованные сущности
if all_ents:
    pd.DataFrame(all_ents, columns=["entity","label"]).to_csv(
        out_dir / "entities.csv", index=False, encoding="utf-8-sig")
    exported.append("entities.csv")


# Czysty tekst dla audiobooka
audio_text = "\n\n".join(corpus)
(out_dir / "чистый_текст_для_аудио.txt").write_text(
    audio_text, encoding="utf-8")
exported.append("чистый_текст_для_аудио.txt")
print(f"Файлы сохранены в папку: {out_dir.resolve()}")
for fname in exported:
    print(f"  {fname}")


## Итоговый отчёт

Ячейка ниже собирает все ключевые результаты анализа в один текстовый отчёт.  
Формат оптимизирован для чтения экранными читалками (NVDA, JAWS): только текст, без цветов, без псевдографики.

In [ ]:
print("=" * 70)
print("СВОДНЫЙ ОТЧЁТ ПО АНАЛИЗУ ТЕКСТА")
print("=" * 70)
print()
_src = SOURCE_FILE if SOURCE_FILE else "встроенный пример"
_model = MODEL_BY_LANG.get(LANG, "неизвестна")
print(f"Исходный файл             : {_src}")
print(f"Определённый язык         : {LANG_NAMES.get(LANG, LANG)} ({LANG})")
print(f"Модель spaCy              : {_model}")
print(f"Количество документов     : {len(corpus)}")
print()
total_words = sum(len(wt) for wt in word_tokens)
total_sents = sum(len(st) for st in sent_tokens)
total_content = sum(len(ft) for ft in filtered_tokens)
print(f"Всего токенов             : {total_words}")
print(f"Всего предложений         : {total_sents}")
print(f"Содержательных слов       : {total_content}")
print(f"Размер списка стоп-слов   : {len(stop_words)}")
print()
print(f"Именованных сущностей     : {len(all_ents)}")
if all_ents:
    for etype, cnt in Counter(label for _, label in all_ents).most_common():
        print(f"  {etype:<12}: {cnt}")
print()
print(f"Словарь BoW (UniGram)     : {len(vocab_uni)} терминов")
print(f"Словарь TF-IDF            : {len(feature_names)} терминов")
print()
print(f"Запрос (авто)             : \"{AUTO_QUERY}\"")
_best = ranked_order[0]
print(f"Лучший результат поиска   : Документ {_best+1} (косинус = {cosine_scores[_best]:.4f})")
print("Ранжирование документов (топ-10):")
_rank_show = min(10, len(ranked_order))
for rank, idx2 in enumerate(ranked_order[:_rank_show], 1):
    print(f"  Место {rank}: Документ {idx2+1} — оценка {cosine_scores[idx2]:.4f}")
if len(ranked_order) > _rank_show:
    print(f"  ... (показаны топ-{_rank_show} из {len(ranked_order)})")
print()
print("Результаты анализа тональности:")
_smap = STAR_MAP if "STAR_MAP" in dir() else {}
for i, (label, score) in enumerate(sentiment_results, 1):
    readable = _smap.get(label, label)
    print(f"  Документ {i}: {readable} (уверенность {score:.2%})")
print()
if corpus_issues:
    fixes = [(d, desc) for d, k, desc in corpus_issues if k == "исправлено"]
    warns = [(d, desc) for d, k, desc in corpus_issues if k == "предупреждение"]
    if fixes:
        print(f"Автоматически исправлено: {len(fixes)}")
        for d, desc in fixes:
            print(f"  Документ {d}: {desc}")
    if warns:
        print(f"\nПредупреждения (ручная проверка): {len(warns)}")
        for d, desc in warns:
            print(f"  Документ {d}: {desc}")
else:
    print("Проблем в тексте не обнаружено.")
print()
print("Тезисы:")
_n_theses = len(df_theses) if "df_theses" in dir() else 0
print(f"  Сформировано тезисов    : {_n_theses}")
if "df_theses" in dir() and _n_theses > 0:
    _tf = THESES_FILE if "THESES_FILE" in dir() else "тезисы.txt"
    print(f"  Сохранены в файл        : {_tf}")
    for _, row in df_theses.head(5).iterrows():
        _pid2 = int(row["para_id"])
        _prev2 = row["sentence"][:80].replace("\n"," ")
        print(f"    Абзац {_pid2:3d}: {_prev2}...")
print()
if "topic_keywords" in dir() and topic_keywords:
    print(f"Тем обнаружено            : {len(topic_keywords)}")
    for t, kws in topic_keywords.items():
        _kw = ", ".join(kws[:5])
        print(f"  Тема {t}: {_kw}")
    print()
print("=" * 70)
print("Экспорт (аудио):")
_audio_path = out_dir / "чистый_текст_для_аудио.txt" if 'out_dir' in dir() else None
if _audio_path and _audio_path.exists():
    _audio_size = _audio_path.stat().st_size
    print(f"  чистый_текст_для_аудио.txt : {_audio_size:,} байт — текст готов к синтезу речи")
else:
    print("  чистый_текст_для_аудио.txt : не найден (запустите ячейку экспорта)")
print()
print("Конец отчёта. Все результаты совместимы с экранными читалками.")
print("=" * 70)


## Интерактивная система вопросов и ответов (RAG)

Простой цикл «вопрос — ответ» на основе TF-IDF и косинусного сходства.  
Введите вопрос — система найдёт 3 наиболее релевантных фрагмента текста из корпуса.

Для выхода введите: **wyjście**, **выход** или **exit**.

In [ ]:
# ============================================================
# Интерактивная система Q&A на основе TF-IDF (RAG)
# Одиночный запрос — выполните ячейку заново для следующего вопроса.
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

EXIT_WORDS = {"wyjście", "выход", "exit"}

# Строим индекс TF-IDF по абзацам (paragraphs)
_qa_texts = paragraphs if 'paragraphs' in dir() and paragraphs else corpus
_qa_stop = list(stop_words) if stop_words else "english"

_qa_tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words=_qa_stop,
    min_df=1
)
try:
    _qa_matrix = _qa_tfidf.fit_transform(_qa_texts)
    print("Система Q&A готова.")
    print(f"Индекс построен по {len(_qa_texts)} фрагментам текста.")
    print("-" * 60)
except Exception as _qa_err:
    print(f"[ОШИБКА] Не удалось построить индекс TF-IDF: {_qa_err}")
    _qa_matrix = None

if _qa_matrix is not None:
    try:
        query = input("Задайте вопрос: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nСессия Q&A завершена.")
        query = ""

    if query and query.lower() not in EXIT_WORDS:
        try:
            _q_vec = _qa_tfidf.transform([query])
            _scores = linear_kernel(_q_vec, _qa_matrix).flatten()
            _top_ids = _scores.argsort()[::-1][:3]

            # Filtrujemy tylko trafienia z score >= 0.001
            relevant = [
                (frag_idx, _scores[frag_idx])
                for frag_idx in _top_ids
                if _scores[frag_idx] >= 0.001
            ]

            print()
            print(f"Вопрос: {query}")
            if relevant:
                print("Наиболее релевантные фрагменты:")
                print("-" * 60)
                for rank_q, (frag_idx, score_q) in enumerate(relevant, 1):
                    fragment = _qa_texts[frag_idx].replace("\n", " ").strip()
                    print(f"{rank_q}. (сходство: {score_q:.4f})")
                    print(f"   {fragment[:400]}")
                    print()
            else:
                print("Совпадений не найдено. Попробуйте другой запрос.")
            print("-" * 60)
        except Exception as _qa_exc:
            print(f"[ОШИБКА] {_qa_exc}")
    elif query.lower() in EXIT_WORDS:
        print("Сессия Q&A завершена. До свидания!")
